# SDE-Net t+1h post-hoc con label STGAN

Questo notebook **non riaddestra SDE-Net o STGAN**. Legge le predizioni del run SDE-Net originale t+1h, sostituisce le label MTGFlow con le decisioni STGAN sulla coppia esatta `(location, timestamp target)` e rigenera la stessa analisi normal/rare.

`event_group` non viene creato. Le righe iniziali non valutate da STGAN per il context window vengono escluse con un inner join e conteggiate nell'audit.

In [ ]:
import json, os, subprocess, sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / 'physiq_pv').is_dir():
    for parent in ROOT.parents:
        if (parent / 'physiq_pv').is_dir():
            ROOT = parent
            break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from physiq_pv.experiments import sde_pipeline as pipe
from physiq_pv.reporting.pointwise_detector_posthoc import build_pointwise_detector_evaluation
print('repo root:', ROOT)

## 1. Percorsi t+1h

Sul server i percorsi possono essere sovrascritti con `STGAN_SEED_DIR`, `SDE_PREDICTIONS_T1` e `STGAN_POSTHOC_ROOT`.

In [ ]:
STGAN_SEED = 20
STGAN_SEED_DIR = Path(os.environ.get(
    'STGAN_SEED_DIR', ROOT / 'outputs' / 'pvgis_stgan' / 'paper_reference' / f'seed_{STGAN_SEED}'
)).resolve()
STGAN_SCORES = STGAN_SEED_DIR / 'anomaly_scores.csv'
BASE_CONFIG = {**pipe.DEFAULT_CONFIG,
    'name': 'paper_faithful_gaussian_detector_mtgflow_ep60',
    'horizon': 1,
    'epochs': 60, 'batch_size': 16, 'lr': 1e-4, 'lr_g': 1e-2,
    'dropout': 0.0, 'train_normal_only': False, 'anomaly_source': 'detector',
    'ood_smoke_test': True, 'sde_sigma_warmup_epochs': 30,
    'irradiance_loss_weight': 0.1, 'detector_regional_quantile': 0.975,
}
assert BASE_CONFIG['horizon'] == 1
SDE_PREDICTIONS = Path(os.environ.get(
    'SDE_PREDICTIONS_T1', ROOT / pipe.make_out_dir(BASE_CONFIG) / 'predictions.csv'
)).resolve()
EVALUATION_DIR = Path(os.environ.get(
    'STGAN_POSTHOC_ROOT', ROOT / 'outputs' / f'sde_stgan_t1_pointwise_seed{STGAN_SEED}'
)).resolve()
MIN_MATCH_FRACTION = 0.90
RUN_RELABEL = True
RUN_ANALYSIS = True
ALLOW_OVERWRITE = False

print('STGAN scores:', STGAN_SCORES)
print('SDE t+1 predictions:', SDE_PREDICTIONS)
print('evaluation output:', EVALUATION_DIR)

In [ ]:
missing = [path for path in (STGAN_SCORES, SDE_PREDICTIONS) if not path.is_file()]
if missing:
    raise FileNotFoundError('File mancanti:\n' + '\n'.join(map(str, missing)))
stgan_header = set(pd.read_csv(STGAN_SCORES, nrows=0).columns)
required_stgan = {'location', 'timestamp', 'anomaly_score', 'threshold', 'is_anomaly'}
if not required_stgan <= stgan_header:
    raise ValueError(f'Colonne STGAN mancanti: {sorted(required_stgan - stgan_header)}')
sde_header = set(pd.read_csv(SDE_PREDICTIONS, nrows=0).columns)
required_sde = {'location', 'timestamp', 'y_true'}
if not required_sde <= sde_header or not ({'y_pred', 'y_pred_mean'} & sde_header):
    raise ValueError(f'Schema SDE-Net t+1 non compatibile: {sorted(sde_header)}')
print('OK: schemi t+1 e STGAN compatibili')

## 2. Join puntuale STGAN → predizioni t+1h

In [ ]:
if RUN_RELABEL:
    RELABEL_RESULT = build_pointwise_detector_evaluation(
        SDE_PREDICTIONS, STGAN_SCORES, EVALUATION_DIR,
        detector_name='stgan', min_match_fraction=MIN_MATCH_FRACTION,
        allow_overwrite=ALLOW_OVERWRITE,
    )
else:
    print("RUN_RELABEL=False: uso l'output evaluation-only esistente.")

metadata_path = EVALUATION_DIR / 'evaluation_source.json'
if not metadata_path.is_file():
    raise FileNotFoundError(metadata_path)
AUDIT = json.loads(metadata_path.read_text(encoding='utf-8'))
display(pd.DataFrame([AUDIT])[[
    'detector', 'source_prediction_rows', 'matched_rows',
    'excluded_unmatched_rows', 'match_fraction', 'normal_rows', 'rare_rows',
]])

In [ ]:
joined_path = EVALUATION_DIR / 'predictions.csv'
columns = set(pd.read_csv(joined_path, nrows=0).columns)
assert 'anomaly_group' in columns
assert 'event_group' not in columns
print('OK: la valutazione t+1 usa soltanto anomaly_group STGAN')

## 3. Analisi post-hoc t+1h

In [ ]:
ANALYSIS_COMMAND = pipe.build_analysis_command(
    str(EVALUATION_DIR), BASE_CONFIG, predictions=str(joined_path),
)
if RUN_ANALYSIS:
    subprocess.run(ANALYSIS_COMMAND, check=True, cwd=ROOT)
else:
    print('RUN_ANALYSIS=False: analisi non avviata.')

In [ ]:
METRICS_BY_LABEL = pd.read_csv(EVALUATION_DIR / 'metrics_by_anomaly_label.csv')
display(METRICS_BY_LABEL)
FIGURE_PATHS = pipe.build_posthoc_figures(str(EVALUATION_DIR))
print({name: str(path) for name, path in FIGURE_PATHS.items()})

## Interpretazione

La valutazione confronta l'errore t+1h sui campioni che STGAN classifica normali o anomali per lo stesso target. Le label sono usate soltanto dopo il forecasting e non modificano il training SDE-Net.